In [1]:
%load_ext autoreload
%autoreload 2

import os
import requests
import pandas as pd
from dotenv import load_dotenv
from gasp import validate

load_dotenv()
API_KEY = os.getenv("EIA_API_KEY")

In [2]:
url = "https://api.eia.gov/v2/natural-gas/stor/wkly/data/"

params = {
    "api_key": API_KEY,
    "frequency": "weekly",
    "data[0]": "value",
    "facets[series][]": "NW2_EPG0_SWO_R48_BCF",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000,
}

r = requests.get(url, params=params)
r.raise_for_status()
rows = r.json()["response"]["data"]
len(rows)

876

In [3]:
df = pd.DataFrame(rows)[["period", "value"]]
df["period"] = pd.to_datetime(df["period"])
df["value"] = pd.to_numeric(df["value"]).astype(float)
df["series"] = "NW2_EPG0_SWO_R48_BCF"
df = df[["period", "series", "value"]].sort_values("period").reset_index(drop=True)
df.dtypes

period    datetime64[us]
series               str
value            float64
dtype: object

In [4]:
change = df["value"].diff()
print(change.describe())
df.assign(change=change).loc[change.abs().nlargest(5).index]

count    875.000000
mean       0.514286
std       98.404053
min     -359.000000
25%      -59.000000
50%       36.000000
75%       76.000000
max      132.000000
Name: value, dtype: float64


,period,series,value,change
418,2018-01-05,NW2_EPG0_SWO_R48_BCF,2767.0,-359.0
839,2026-01-30,NW2_EPG0_SWO_R48_BCF,2454.0,-359.0
581,2021-02-19,NW2_EPG0_SWO_R48_BCF,1943.0,-338.0
733,2024-01-19,NW2_EPG0_SWO_R48_BCF,2856.0,-326.0
786,2025-01-24,NW2_EPG0_SWO_R48_BCF,2571.0,-321.0


In [5]:
validate.validate_storage(df)

[]

In [6]:
def show(name, bad):
    print(name)
    problems = validate.validate_storage(bad)
    for p in problems:
        print("   ", p)
    if not problems:
        print("    (no problems found)")

## Broken cases
Each case starts from `df.copy()` so the clean data stays clean.
Some cases correctly report more than one problem:
- **3:** moving a date to Thursday also leaves its Friday missing
- **7:** a value of -50 also creates huge jumps in and out of that week
- **8:** a +1000 spike creates two jumps, up into the week and back down after it

In [7]:
# 1. missing column
bad = df.copy().drop(columns="series")
show("1. missing column", bad)

# 2. wrong type
bad = df.copy()
bad["value"] = bad["value"].astype(str)
show("2. value stored as text", bad)

# 3. date moved to Thursday
bad = df.copy()
bad.loc[100, "period"] = bad.loc[100, "period"] - pd.Timedelta(days=1)
show("3. date moved to Thursday", bad)

# 4. duplicate row
bad = pd.concat([df.copy(), df.iloc[[100]]]).sort_values("period").reset_index(drop=True)
show("4. duplicated row", bad)

# 5. dropped week
bad = df.copy().drop(index=100).reset_index(drop=True)
show("5. dropped week", bad)

# 6. missing value
bad = df.copy()
bad.loc[100, "value"] = float("nan")
show("6. value set to NaN", bad)

# 7. negative storage
bad = df.copy()
bad.loc[100, "value"] = -50.0
show("7. negative storage", bad)

# 8. impossible jump
bad = df.copy()
bad.loc[100, "value"] = bad.loc[100, "value"] + 1000
show("8. impossible jump", bad)

# 9. not sorted oldest first
bad = df.copy().iloc[::-1].reset_index(drop=True)
show("9. sorted newest first", bad)

1. missing column
    missing column: series
2. value stored as text
    wrong type: value is str, expected float
3. date moved to Thursday
    not a Friday: 2011-12-01 (Thursday)
    missing week: 2011-12-02 (NW2_EPG0_SWO_R48_BCF)
4. duplicated row
    duplicate row: 2011-12-02 (NW2_EPG0_SWO_R48_BCF)
5. dropped week
    missing week: 2011-12-02 (NW2_EPG0_SWO_R48_BCF)
6. value set to NaN
    missing value: value at 2011-12-02
7. negative storage
    negative storage: -50 Bcf at 2011-12-02
    impossible jump: -3900 Bcf into 2011-12-02 (limit 500) (NW2_EPG0_SWO_R48_BCF)
    impossible jump: +3779 Bcf into 2011-12-09 (limit 500) (NW2_EPG0_SWO_R48_BCF)
8. impossible jump
    impossible jump: +980 Bcf into 2011-12-02 (limit 500) (NW2_EPG0_SWO_R48_BCF)
    impossible jump: -1101 Bcf into 2011-12-09 (limit 500) (NW2_EPG0_SWO_R48_BCF)
9. sorted newest first
    not sorted by period: 2026-10-02 comes after a later date
